# 詩織 LoRA v3 テスト生成専用ノートブック（Kaggle）

学習済みLoRA（notebook536232ebfd Version 1 の出力 shiori_oneesan_v3.safetensors）を **Add Input** で接続して使う。
学習は行わず、推論だけを実行する。学習ノート内蔵のテストセルは diffusers の LoRA 読み込みで IndexError になったため、v1で実績のある単独推論方式（素の環境＋最新diffusers）で再実行する。
生成は2枚：①比較用の標準プロンプト（髪の記述をv3仕様に強化）、②髪チェック用（肩に流した髪のアップ）。

In [ ]:
# GPU確認
!nvidia-smi


In [ ]:
# 依存関係の用意（torchao は古い版が入っていると peft が ImportError になるため外す）
!pip install -q diffusers transformers accelerate peft huggingface_hub
!pip uninstall -y torchao

# ベースモデル（Animagine XL 4.0）を取得
from huggingface_hub import hf_hub_download
import shutil, os
p = hf_hub_download(repo_id='cagliostrolab/animagine-xl-4.0', filename='animagine-xl-4.0.safetensors')
shutil.copy(p, '/kaggle/working/animagine-xl-4.0.safetensors')
print('model ready:', os.path.getsize('/kaggle/working/animagine-xl-4.0.safetensors'))


In [ ]:
# LoRA を読み込んでテスト生成（2枚）
import glob, os, torch
from diffusers import StableDiffusionXLPipeline
cands = [c for c in glob.glob('/kaggle/input/**/shiori_oneesan_v3.safetensors', recursive=True)]
print('lora candidates:', cands)
assert cands, 'shiori_oneesan_v3.safetensors not found in /kaggle/input'
lora_path = cands[0]
print('lora size:', os.path.getsize(lora_path))
pipe = StableDiffusionXLPipeline.from_single_file('/kaggle/working/animagine-xl-4.0.safetensors', torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights(os.path.dirname(lora_path), weight_name='shiori_oneesan_v3.safetensors')
neg = 'lowres, bad anatomy, bad hands, extra fingers, worst quality, blurry, multiple girls, 2girls, extra person'
os.makedirs('/kaggle/working/output', exist_ok=True)

prompt1 = 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, purple eyes, gentle smile, pink cardigan, shopping street, cafe terrace, sunny day'
g = torch.Generator(device='cuda').manual_seed(42)
img = pipe(prompt1, negative_prompt=neg, width=1024, height=1024, num_inference_steps=28, guidance_scale=7, generator=g).images[0]
img.save('/kaggle/working/output/test_outing_v3.png')
print('saved test_outing_v3.png')

prompt2 = 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, purple eyes, gentle smile, hair over shoulder, white blouse, indoors, window, daylight, close-up portrait'
g = torch.Generator(device='cuda').manual_seed(42)
img2 = pipe(prompt2, negative_prompt=neg, width=1024, height=1024, num_inference_steps=28, guidance_scale=7, generator=g).images[0]
img2.save('/kaggle/working/output/test_hair_v3.png')
print('saved test_hair_v3.png')
img
